# Module 2.3 -- Stateful Agentic Orchestration & Memory Architectures

**Track:** Backend Engineering for AI Applications  
**Toolchain:** LangGraph - Valkey/Redis - pgvector - Temporal  
**Objective:** Build multi-agent systems with persistent memory, plan-and-execute workflows, and deterministic orchestration boundaries.

---

## Why Single-Prompt Agents Fail in Production

**Simple agent:** Give LLM a prompt, let it loop until done.  
**Problem:** It can loop forever, spend $500 in API calls, or hallucinate actions.

### The Critical Rule: Separate Reasoning from Control

| | Orchestrator (Deterministic) | LLM (Stochastic) |
|---|---|---|
| Role | Controls execution flow | Provides reasoning |
| Budget tracking | Enforces cost limits | Unaware of costs |
| Loop control | Max iterations, timeouts | Would loop forever |
| Failure recovery | Retries, fallbacks | Cannot self-recover |
| Tool selection | Validates before execution | Suggests which tool |

### Anti-Pattern: Control Flow in Prompts

```python
# BAD: The LLM decides when to stop (infinite loop risk!)
prompt = "Keep researching until you have a complete answer"

# GOOD: The orchestrator enforces limits
for step in range(MAX_STEPS):    # Deterministic limit
    action = llm.decide(state)   # LLM only reasons
    result = execute(action)     # Orchestrator controls execution
    if is_complete(result): break
```

### Agentic Patterns

| Pattern | Description | When to Use |
|---------|------------|-------------|
| **ReAct** | Think-Act-Observe loop | Simple tool-using agents |
| **Plan-and-Execute** | Plan all steps first, then execute | Complex multi-step tasks |
| **Reflection** | Agent reviews its own output | Quality-critical tasks |
| **Hierarchical** | Manager agent delegates to specialists | Multi-domain problems |


### 🎓 Junior to Senior: Concept Bridge

**The Senior Perspective:** Simple linear chains fall apart when reasoning requires loops, backtracking, and memory. Seniors use precise State Graph architectures (LangGraph) to orchestrate complex, resilient agent workflows.

**Mental Model:** A Chain is a one-way street; it can't turn back. An Agentic Graph is a flowchart with 'If/Else' logic, memory, and loops, allowing the LLM to rethink its approach if the first tool call fails.

**Common Junior Pitfall:** Giving an LLM 45 tools at once and telling it to 'figure it out' (AutoGPT style), resulting in infinite loops. Seniors constrain agents into strict, highly-defined State Machines.

---


---

## Prerequisite: Concurrency for Beginners

Multi-agent systems run MULTIPLE agents simultaneously. Before diving in, you need three concepts:

### 1. Race Condition
Two agents read the same counter, both increment it, both write back. Expected: +2. Actual: +1 (one write overwrites the other).

```
Agent A reads counter = 10
Agent B reads counter = 10     <- reads BEFORE A writes
Agent A writes counter = 11
Agent B writes counter = 11    <- overwrites A's write!
Expected: 12, Got: 11
```

### 2. Locks
A lock is a "bathroom occupied" sign. Only one agent can hold the lock at a time.

```python
lock.acquire()        # "Occupied" sign goes up
counter = read()      # Only I can read
counter += 1
write(counter)        # Only I can write
lock.release()        # "Vacant" -- next agent can enter
```

### 3. Eventual Consistency
In distributed systems, different nodes may temporarily show different values. Like two ATMs showing different balances for 5 seconds after a transfer -- but they EVENTUALLY agree.

**Why this matters here:** The dual-tier memory (Redis + pgvector) is eventually consistent. An agent might read slightly stale data from the vector DB. The code must tolerate this.


---

## Production Reality Check

| In This Notebook | In Production |
|---|---|
| `HotMemory` (Python dict) | Redis/Valkey cluster (3+ nodes, replication) |
| `WarmMemory` (NumPy arrays) | pgvector extension on PostgreSQL with indexes |
| `PlanAndExecuteAgent` (simulated) | LangGraph + Temporal for durable workflows |
| Sequential execution | Async concurrent agents with shared state locks |

### Real Multi-Agent Deployment

| Component | Technology | Why |
|-----------|-----------|-----|
| Workflow engine | Temporal | Durable execution, automatic retries |
| State graph | LangGraph | Defines agent transitions |
| Session memory | Redis Cluster | Sub-millisecond reads |
| Long-term memory | pgvector (PostgreSQL) | SQL joins + vector search |
| Monitoring | Langfuse | Trace every agent decision |
| Cost control | Budget middleware | Hard limits per agent per task |


In [ ]:
# Cell 1 -- Install
!pip install -q redis numpy chromadb


---
## 1 - Plan-and-Execute Agent

### How It Works

```
User Request -> [PLANNER] -> Step 1, Step 2, Step 3
                                |       |       |
                            [EXECUTE] [EXECUTE] [EXECUTE]
                                |       |       |
                            [VERIFY] -- if failed --> [REPLAN]
```

The PLANNER (LLM) creates a list of steps.  
The EXECUTOR (deterministic code) runs each step with safety checks.  
If a step fails, the planner is asked to REPLAN from the current state.


In [ ]:
# Cell 2 -- Plan-and-Execute implementation
import json, time

class PlanAndExecuteAgent:
    def __init__(self, max_steps=10, max_cost_usd=1.0):
        self.max_steps = max_steps
        self.max_cost = max_cost_usd
        self.total_cost = 0.0
        self.execution_log = []

    def plan(self, task):
        """LLM generates a plan (simulated)."""
        self.total_cost += 0.01  # Track cost
        plans = {
            'research_competitor': ['search_web("competitor analysis AI market")', 'extract_key_metrics(results)', 'summarize_findings(metrics)'],
            'prepare_report': ['query_database("SELECT * FROM sales")', 'calculate_trends(data)', 'generate_charts(trends)', 'compile_report(charts)'],
        }
        return plans.get(task, ['search_web(task)', 'summarize(results)'])

    def execute_step(self, step, step_num):
        """Orchestrator executes with safety checks."""
        if self.total_cost >= self.max_cost:
            return {'status': 'BLOCKED', 'reason': f'Cost limit ${self.max_cost} reached'}
        if step_num >= self.max_steps:
            return {'status': 'BLOCKED', 'reason': f'Max steps {self.max_steps} reached'}

        self.total_cost += 0.005
        success = step_num < 5  # Simulate: first 5 steps succeed
        result = {'status': 'OK' if success else 'FAILED', 'step': step, 'cost_so_far': f'${self.total_cost:.3f}'}
        self.execution_log.append(result)
        return result

    def run(self, task):
        print(f'Task: "{task}"')
        plan = self.plan(task)
        print(f'Plan: {len(plan)} steps')
        for i, step in enumerate(plan):
            result = self.execute_step(step, i)
            status_icon = '[OK]' if result['status'] == 'OK' else '[!!]'
            print(f'  Step {i+1}: {status_icon} {step} (cost: {result.get("cost_so_far", "N/A")})')
            if result['status'] != 'OK':
                print(f'    Reason: {result.get("reason", "Step failed")}')
                break
        print(f'Done. Total cost: ${self.total_cost:.3f}')

agent = PlanAndExecuteAgent(max_steps=10, max_cost_usd=0.50)
agent.run('research_competitor')
print()
agent.run('prepare_report')


---
## 2 - Dual-Tier Memory Architecture

### Why Two Memory Tiers?

| Tier | Technology | Latency | Durability | What It Stores |
|------|-----------|---------|-----------|----------------|
| **Tier 1** (Hot) | Valkey/Redis | <1ms | Session-lived | Current conversation, tool state, checkpoints |
| **Tier 2** (Warm) | pgvector/Pinecone | 5-50ms | Permanent | Long-term facts, episodic memory, user preferences |

### When to Use Which Vector Database?

| Database | Max Vectors | Strength | Best For |
|----------|------------|----------|----------|
| **pgvector** | ~100M | SQL joins with metadata | When you need vectors + relational data |
| **Pinecone** | Billions | Managed, zero-ops | When you don't want to manage infrastructure |
| **Milvus** | Billions | Self-hosted, GPU-accelerated | High-throughput, on-premise |
| **Qdrant** | Billions | Advanced filtering | When you need complex metadata filters |
| **ChromaDB** | ~10M | Simplest to start | Prototyping, small datasets |


In [ ]:
# Cell 3 -- Dual-tier memory implementation
import json, time, numpy as np

class HotMemory:
    """Tier 1: Fast session state (simulates Redis/Valkey)."""
    def __init__(self):
        self.store = {}
        self.ttl = {}  # time-to-live

    def set(self, key, value, ttl_seconds=3600):
        self.store[key] = value
        self.ttl[key] = time.time() + ttl_seconds

    def get(self, key):
        if key in self.store and time.time() < self.ttl.get(key, 0):
            return self.store[key]
        return None

class WarmMemory:
    """Tier 2: Long-term semantic memory (simulates pgvector)."""
    def __init__(self, dim=384):
        self.memories = []
        self.dim = dim

    def store(self, text, metadata=None):
        embedding = np.random.randn(self.dim)  # In production: model.encode(text)
        self.memories.append({'text': text, 'embedding': embedding, 'metadata': metadata or {}})

    def search(self, query, top_k=3):
        query_emb = np.random.randn(self.dim)
        scored = []
        for m in self.memories:
            sim = np.dot(query_emb, m['embedding']) / (np.linalg.norm(query_emb) * np.linalg.norm(m['embedding']) + 1e-10)
            scored.append((sim, m['text']))
        return sorted(scored, key=lambda x: -x[0])[:top_k]

# Demo
hot = HotMemory()
warm = WarmMemory()

# Session state (Tier 1)
hot.set('session:user123:context', {'current_task': 'report', 'steps_completed': 2, 'budget_remaining': 0.45})
hot.set('session:user123:tools_used', ['query_database', 'search_web'])

# Long-term memory (Tier 2)
warm.store('User prefers concise answers under 200 words', {'type': 'preference'})
warm.store('Last quarter revenue was $52.8M, exceeding projections', {'type': 'fact', 'date': '2025-10-15'})
warm.store('User is working on competitor analysis project', {'type': 'context'})

print('Tier 1 (Hot Memory - Redis/Valkey):')
ctx = hot.get('session:user123:context')
print(f'  Session context: {ctx}')
print(f'  Tools used: {hot.get("session:user123:tools_used")}')

print(f'\nTier 2 (Warm Memory - pgvector):')
results = warm.search('What does the user prefer?')
for score, text in results:
    print(f'  [{score:.3f}] {text}')


---
## 3 - Emergence in Multi-Agent Systems

### What is Emergence?

When multiple agents interact, **unexpected behaviors** can appear that weren't programmed.

| Type | Example | Dangerous? |
|------|---------|------------|
| **Positive** | Agents specialize automatically | No (desirable) |
| **Positive** | One agent catches another's errors | No (desirable) |
| **Negative** | Agents game each other's metrics | Yes |
| **Negative** | Information hoarding between agents | Yes |
| **Negative** | Cascade failure (one fails, all fail) | Yes (critical) |

### How to Prevent Negative Emergence

1. **Shared state**: All agents read from the same state store (no hidden information)
2. **Circuit breakers**: If an agent loops >5 times, trip the breaker
3. **Budget limits**: Hard cost caps per agent per task
4. **Human checkpoints**: Require approval before irreversible actions


---
## Summary

| Concept | What You Learned |
|---------|------------------|
| Orchestrator vs LLM | Deterministic control vs stochastic reasoning |
| Plan-and-Execute | Plan first, execute with safety checks |
| Dual-tier memory | Hot (Redis) + Warm (pgvector) |
| Emergence | Unexpected behaviors in multi-agent systems |

**Next -->** `17_ai_gateways_resilience.ipynb` -- AI Gateways, Caching & Circuit Breakers